# 🤖 Notebook 2 — Model Training
**Crop Disease Diagnostician** | MobileNetV2 Fine-Tuning

**Prerequisite**: Run `01_data_prep.ipynb` first in this same Colab session.
Data must be in `/content/data/train/` and `/content/data/val/`.

If you've disconnected since running notebook 1:
- Re-run notebook 1 to re-download and re-organize the data
- Or mount Google Drive and load from there

**Expected runtime**: ~45–60 min on T4 GPU.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json, os

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

# Mixed precision for faster T4 training
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Mixed precision: mixed_float16')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42
DATA_DIR    = '/content/data'
MODEL_DIR   = '/content/model'

# Phase 1: train head only (base frozen)
PHASE1_EPOCHS = 5
PHASE1_LR     = 1e-3

# Phase 2: fine-tune top layers
PHASE2_EPOCHS        = 15
PHASE2_LR            = 1e-4
UNFREEZE_FROM_LAYER  = 100   # unfreeze MobileNetV2 from this layer index

os.makedirs(MODEL_DIR, exist_ok=True)

# Load labels from notebook 1 output
with open(f'{MODEL_DIR}/labels.json') as f:
    LABELS_MAP = json.load(f)
NUM_CLASSES = len(LABELS_MAP)

print(f'Classes: {NUM_CLASSES}')
for i, k in enumerate(LABELS_MAP):
    print(f'  [{i}] {k}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA PIPELINE
# image_dataset_from_directory assigns labels alphabetically by folder name.
# Our folder names ARE our class keys (e.g. 'tomato_early_blight'),
# and LABELS_MAP is also sorted alphabetically, so indices match perfectly.
# ─────────────────────────────────────────────────────────────────────────────

AUTOTUNE = tf.data.AUTOTUNE

# Verify folder structure
train_classes = sorted(os.listdir(f'{DATA_DIR}/train'))
print(f'Train folders: {train_classes}')
assert train_classes == LABELS_MAP, \
    f'Mismatch! Folders: {train_classes}\nExpected: {LABELS_MAP}'
print('✅ Folder names match LABELS_MAP — label assignment is correct')

def make_dataset(split, augment=False):
    """Load images and apply MobileNetV2 preprocessing."""
    ds = tf.keras.utils.image_dataset_from_directory(
        f'{DATA_DIR}/{split}',
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=None,          # we batch after preprocessing
        label_mode='categorical', # one-hot labels
        shuffle=(split == 'train'),
        seed=SEED,
        class_names=LABELS_MAP,   # enforce alphabetical ordering
    )

    def preprocess(image, label):
        # Augmentation (training only)
        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
            image = tf.image.random_brightness(image, 0.2)
            image = tf.image.random_contrast(image, 0.8, 1.2)
            image = tf.image.random_saturation(image, 0.8, 1.2)
            # Crop 85% then resize back (simulates slight rotation/zoom)
            image = tf.image.random_crop(image, [int(IMG_SIZE*0.85), int(IMG_SIZE*0.85), 3])
            image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
        # MobileNetV2 preprocessing: [0,255] → [-1, 1]
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        return image, label

    return (
        ds
        .map(preprocess, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

ds_train = make_dataset('train', augment=True)
ds_val   = make_dataset('val',   augment=False)
ds_test  = make_dataset('test',  augment=False)

# Quick check
images, labels = next(iter(ds_train))
print(f'\nBatch shape: images={images.shape}  labels={labels.shape}')
print(f'Pixel range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]  (expect ~-1 to 1)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD CLASS WEIGHTS (computed in notebook 1)
# ─────────────────────────────────────────────────────────────────────────────

with open(f'{MODEL_DIR}/split_info.json') as f:
    split_info = json.load(f)

class_weights = {int(k): v for k, v in split_info['class_weights'].items()}
print('Class weights:', {LABELS_MAP[i]: round(w, 3) for i, w in class_weights.items()})

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MODEL: MobileNetV2 + custom classification head
# ─────────────────────────────────────────────────────────────────────────────

def build_model(num_classes, trainable_base=False):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = trainable_base

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=trainable_base)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    # Cast to float32 before softmax (mixed precision safety)
    x = tf.cast(x, tf.float32)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32')(x)

    return tf.keras.Model(inputs, outputs), base

model, base_model = build_model(NUM_CLASSES, trainable_base=False)

trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
total_params     = sum(tf.size(w).numpy() for w in model.weights)
print(f'Trainable params: {trainable_params:,} / Total: {total_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: TRAIN HEAD (base frozen, 5 epochs)
# ─────────────────────────────────────────────────────────────────────────────

model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'],
)

cb1 = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{MODEL_DIR}/best_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1),
]

print(f'Phase 1: training head only for {PHASE1_EPOCHS} epochs...')
hist1 = model.fit(
    ds_train,
    epochs=PHASE1_EPOCHS,
    validation_data=ds_val,
    class_weight=class_weights,
    callbacks=cb1,
    verbose=1,
)
print(f'Best phase 1 val accuracy: {max(hist1.history["val_accuracy"]):.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: FINE-TUNE (unfreeze top layers, low LR, up to 15 more epochs)
# ─────────────────────────────────────────────────────────────────────────────

base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM_LAYER]:
    layer.trainable = False

n_unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen {n_unfrozen} base layers (from index {UNFREEZE_FROM_LAYER}+)')

model.compile(
    optimizer=tf.keras.optimizers.Adam(PHASE2_LR),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy'],
)

cb2 = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{MODEL_DIR}/crop_disease_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, verbose=1),
]

print(f'Phase 2: fine-tuning for up to {PHASE2_EPOCHS} more epochs...')
hist2 = model.fit(
    ds_train,
    initial_epoch=PHASE1_EPOCHS,
    epochs=PHASE1_EPOCHS + PHASE2_EPOCHS,
    validation_data=ds_val,
    class_weight=class_weights,
    callbacks=cb2,
    verbose=1,
)
print(f'Best phase 2 val accuracy: {max(hist2.history["val_accuracy"]):.4f}')

# Also save as .h5 for compatibility with tensorflowjs_converter
model.save(f'{MODEL_DIR}/crop_disease_model.h5')
print(f'Model saved: {MODEL_DIR}/crop_disease_model.h5')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TRAINING CURVES
# ─────────────────────────────────────────────────────────────────────────────

all_acc      = hist1.history['accuracy']     + hist2.history['accuracy']
all_val_acc  = hist1.history['val_accuracy'] + hist2.history['val_accuracy']
all_loss     = hist1.history['loss']         + hist2.history['loss']
all_val_loss = hist1.history['val_loss']     + hist2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, train, val, title in [
    (ax1, all_acc, all_val_acc, 'Accuracy'),
    (ax2, all_loss, all_val_loss, 'Loss'),
]:
    ax.plot(train, label='Train', color='#16a34a', lw=2)
    ax.plot(val,   label='Val',   color='#f59e0b', lw=2)
    ax.axvline(x=PHASE1_EPOCHS - 0.5, color='gray', ls='--', label='Fine-tune start')
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

# Quick val evaluation
val_loss, val_acc = model.evaluate(ds_val, verbose=0)
print(f'\nFinal val accuracy: {val_acc*100:.2f}%')
print('\n✅ Notebook 2 complete! Proceed to 03_convert_tflite.ipynb')